In [2]:
import torch
from dinosaw.utils import do_2D_pca
from dinosaw.helpers import ModelTypes, model_names, get_models, get_features

import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import minmax_scale, scale
from os import listdir

from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib import font_manager


SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

flash attention installed


In [3]:
def add_custom_font(font_folder: str, font_name: str="Grotesk") -> None:
    try:
        font_paths = (f'{font_folder}/{font_name}.ttf', f'{font_folder}/{font_name}-Bold.ttf')
        for font_path in font_paths:
            font_manager.fontManager.addfont(font_path)
        prop = font_manager.FontProperties(fname=font_paths[0])

        plt.rcParams['font.family'] = 'sans-serif'
        plt.rcParams['font.sans-serif'] = prop.get_name()
    except Exception as e:
        print(f"Can't load custom font: {e} ")

In [4]:
def get_shared_pca(features_: list[np.ndarray], fg_masks: list[np.ndarray]) -> PCA:
    all_features = []
    for feature, fg_mask in zip(features_, fg_masks):
        masked = feature[fg_mask]
        all_features.append(masked)
        print(feature.shape, fg_mask.shape, masked.shape)
    all_features_concat = np.concatenate(all_features, axis=0)
    all_features_scaled = scale(all_features_concat, axis=0)
    pca = PCA(n_components=3)
    pca.fit(all_features_scaled)
    return pca

def apply_shared_pca(features: np.ndarray, fg_mask: np.ndarray, pca: PCA) -> np.ndarray:
    h, w, c = features.shape
    valid_features = features[fg_mask]
    features_scaled = scale(valid_features, axis=0)
    emb_3d = pca.transform(features_scaled)
    emb_3d_minmax = minmax_scale(emb_3d, feature_range=(0, 1), axis=0)

    out = np.zeros((h, w, 3), dtype=np.float32)
    out[fg_mask] = emb_3d_minmax
    return out

In [5]:
images: list[Image.Image] = []
path = 'data/shared_pca/'
folder_name = 'bison'

for file_name in listdir(f'{path}/{folder_name}'):
    img = Image.open(f'{path}/{folder_name}/{file_name}').convert('RGB')

    shortest = min(img.size)
    L = 384
    sf = L / shortest
    new_size = (int(img.size[0] * sf), int(img.size[1] * sf))
    img = img.resize(new_size, resample=Image.LANCZOS)

    images.append(img)

In [6]:
enabled_models: tuple[ModelTypes, ...] = ('dv2', 'alibi_dv2_cb')
models = get_models(enabled_models, '../../trained_models', DEVICE, False)

In [7]:

features = {k: [] for k in enabled_models}
features_reduced = {k: [] for k in enabled_models}
for img in images:
    for name, model in models.items():
        feat = get_features(model, img)
        reduced = do_2D_pca(feat, 3, pre_norm='std', post_norm='minmax')
        features[name].append(feat)
        features_reduced[name].append(reduced)

In [8]:
def _get(red_li, img_idx, ch, thr) -> np.ndarray:
    return red_li[img_idx][: , :, ch] > thr



In [9]:
# bison
red = features_reduced[enabled_models[-1]]
fg_masks =[~_get(red, 0, 1, 0.6), ~_get(red, 1, 0, 0.3), ~_get(red, 2, 0, 0.3), ~_get(red, 3, 0, 0.4)]

selected_masks = fg_masks

In [10]:
# plane-birds
# r
# dv2_fg_masks = [reduced_features[0][: , :, 1] > 0.7, reduced_features[1][: , :, 0] < 0.3, reduced_features[2][: , :, 0]  < 0.3, reduced_features[3][: , :, 0] > 0.6]
# alibi_fg_masks = [reduced_features[0][: , :, 0] < 0.5, reduced_features[1][: , :, 0] < 0.45, reduced_features[2][: , :, 0]  < 0.35, reduced_features[3][: , :, 0] < 0.4]

# selected_masks = alibi_fg_masks


# red = features_reduced['alibi']
# fg_masks =[_get(red, 0, 0, 0.7), ~_get(red, 1, 0, 0.6), ~_get(red, 2, 0, 0.3), ~_get(red, 3, 0, 0.6)]

# selected_masks = fg_masks

In [11]:
# # elephants
# dv2_fg_masks = [reduced_features[0][: , :, 1] > 0.7, reduced_features[1][: , :, 0] < 0.3, reduced_features[2][: , :, 0]  < 0.3, reduced_features[3][: , :, 0] > 0.6]
# alibi_fg_masks = [reduced_features[0][: , :, 0] > 0.5, reduced_features[1][: , :, 0] > 0.45, reduced_features[2][: , :, 0]  > 0.35, reduced_features[3][: , :, 0] < 0.3]

# selected_masks = alibi_fg_masks

In [12]:
%%capture
# fig, axs = plt.subplots(len(images), 2, figsize=(12, 8))

# for i, (img, reduced, mask) in enumerate(zip(images, reduced_features, selected_masks)):
#     axs[i, 0].imshow(img)
#     axs[i, 0].axis('off')
#     axs[i, 0].set_title(f'Image {i+1}', fontsize=14)

#     axs[i, 1].imshow(reduced * mask[:, :, np.newaxis])
#     axs[i, 1].axis('off')
#     axs[i, 1].set_title(f'PCA Reduced Features {i+1}', fontsize=14)
# plt.tight_layout()

In [13]:
%%capture
dv2_feats_tr = [f.transpose(1, 2, 0) for f in features[enabled_models[0]]]
dv2_shared_pca = get_shared_pca(dv2_feats_tr, fg_masks)

alibi_feats_tr = [f.transpose(1, 2, 0) for f in features[enabled_models[-1]]]
alibi_shared_pca = get_shared_pca(alibi_feats_tr, selected_masks)

dv2_fg_feats_reduced = [apply_shared_pca(feats, mask, dv2_shared_pca) for feats, mask in zip(dv2_feats_tr, selected_masks)]
alibi_fg_feats_reduced = [apply_shared_pca(feats, mask, alibi_shared_pca) for feats, mask in zip(alibi_feats_tr, selected_masks)]

In [23]:
%%capture
W, H = 3, 2.5
FS = 17
add_custom_font('resources/fonts', 'Grotesk')
fig, axs = plt.subplots(len(images), 3, figsize=(W * len(models), H * 2.3))

for i, (img, dv2_reduced, alibi_reduced, mask) in enumerate(zip(images, dv2_fg_feats_reduced, alibi_fg_feats_reduced, selected_masks)):
    axs[i, 0].imshow(img)
    axs[i, 0].axis('off')

    axs[i, 1].imshow(dv2_reduced * mask[:, :, np.newaxis])
    axs[i, 1].axis('off')
    

    axs[i, 2].imshow(alibi_reduced * mask[:, :, np.newaxis])
    axs[i, 2].axis('off')

    if i == 0:
        axs[i, 1].set_title('DINOv2', fontsize=FS)
        axs[i, 2].set_title('ALiBi-Dv2', fontsize=FS, weight=700)
plt.tight_layout()
plt.savefig('saved/03b.png', dpi=300)